# Generate Synthetic Writing Elements

Generate clean, synthetic writing element glyphs by rendering characters with various fonts
and applying realistic augmentations (rotation, scale, brightness, noise).

**Why synthetic?**
- Full control over quality and volume
- No OCR misclassification errors
- Deterministic reproducibility
- Can generate unlimited samples

**Approach:**
1. Use PIL to render each character with multiple fonts
2. Apply augmentations (rotation ±35°, scale 0.8–1.2, brightness, noise)
3. Organize by category: lowercase, uppercase, punctuation
4. Save as individual 32×32 grayscale PNGs
5. Zip for upload to training notebook

**Output:** `writing_elements_classifier.zip` ready for NotAFigurine training.

## Step 1 — Install dependencies

In [ ]:
!pip install -q pillow numpy matplotlib
!apt-get install -qq fonts-dejavu fonts-liberation  # Additional fonts

## Step 2 — Configuration

In [ ]:
import os
import string
import zipfile
import shutil
import random
from collections import defaultdict

import numpy as np
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt

# ── CHARACTERS TO GENERATE ────────────────────────────────────────
LOWERCASE = string.ascii_lowercase
UPPERCASE = string.ascii_uppercase
PUNCTUATION = '.,!?;:\'"()[]{}/-—–_$£€%*~&^><ç#@'

# ── GENERATION PARAMETERS ─────────────────────────────────────────
IMG_SIZE = 32  # Output glyph size (match training pipeline)
FONT_SIZE = 24  # Approximate font size for rendering
AUGMENTATIONS_PER_CHAR = 10  # Variations per character
MAX_ROTATION_DEG = 35

# Output
OUTPUT_DIR = '/content/writing_elements_glyphs'
ZIP_PATH = '/content/writing_elements_classifier.zip'

print(f'Characters to generate:')
print(f'  Lowercase: {len(LOWERCASE)} ({LOWERCASE})')
print(f'  Uppercase: {len(UPPERCASE)} ({UPPERCASE})')
print(f'  Punctuation: {len(PUNCTUATION)} ({PUNCTUATION})')
print(f'  Total: {len(LOWERCASE) + len(UPPERCASE) + len(PUNCTUATION)} chars')
print(f'\nAugmentations per char: {AUGMENTATIONS_PER_CHAR}')
print(f'Expected glyphs: {(len(LOWERCASE) + len(UPPERCASE) + len(PUNCTUATION)) * AUGMENTATIONS_PER_CHAR:,}')

## Step 3 — Setup fonts and output directories

In [ ]:
# Try to find available fonts on the system
FONT_PATHS = []
font_search_paths = [
    '/usr/share/fonts/truetype/dejavu/',
    '/usr/share/fonts/truetype/liberation/',
    '/usr/share/fonts/truetype/liberation2/',
]

for path in font_search_paths:
    if os.path.exists(path):
        for font_file in os.listdir(path):
            if font_file.endswith('.ttf'):
                FONT_PATHS.append(os.path.join(path, font_file))

print(f'Found {len(FONT_PATHS)} fonts: {[os.path.basename(f) for f in FONT_PATHS[:5]]}...')

if not FONT_PATHS:
    print('⚠️  No TTF fonts found, will use default PIL font')

# Create output directories
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR)

for cat in ['lowercase', 'uppercase', 'punctuation']:
    os.makedirs(os.path.join(OUTPUT_DIR, cat), exist_ok=True)

print(f'\n✅ Output directories created in {OUTPUT_DIR}')

## Step 4 — Character rendering and augmentation

In [ ]:
def render_char(char, font_path=None, font_size=FONT_SIZE):
    """
    Render a single character as a PIL Image.
    Returns a PIL Image with the character centered.
    """
    # Create canvas
    img = Image.new('L', (IMG_SIZE * 2, IMG_SIZE * 2), color=255)  # White background
    draw = ImageDraw.Draw(img)
    
    # Load font
    try:
        if font_path and os.path.exists(font_path):
            font = ImageFont.truetype(font_path, font_size)
        else:
            font = ImageFont.load_default()
    except Exception:
        font = ImageFont.load_default()
    
    # Get text bounding box
    bbox = draw.textbbox((0, 0), char, font=font)
    text_width = bbox[2] - bbox[0]
    text_height = bbox[3] - bbox[1]
    
    # Center the character
    x = (IMG_SIZE * 2 - text_width) // 2
    y = (IMG_SIZE * 2 - text_height) // 2
    
    # Draw character
    draw.text((x, y), char, fill=0, font=font)  # Black text on white
    
    # Crop to center
    img = img.crop((IMG_SIZE // 2, IMG_SIZE // 2, IMG_SIZE // 2 + IMG_SIZE, IMG_SIZE // 2 + IMG_SIZE))
    return img

def augment_char(img, augmentation_index):
    """
    Apply random augmentations to a character image.
    Returns a PIL Image.
    """
    arr = np.array(img, dtype=np.float32) / 255.0
    
    # Brightness
    brightness = random.uniform(0.8, 1.2)
    arr = np.clip(arr * brightness, 0.0, 1.0)
    
    # Rotation
    pil_img = Image.fromarray((arr * 255).astype(np.uint8))
    pil_img = pil_img.rotate(random.uniform(-MAX_ROTATION_DEG, MAX_ROTATION_DEG),
                             resample=Image.BICUBIC, fillcolor=255)
    arr = np.array(pil_img, dtype=np.float32) / 255.0
    
    # Scale
    scale = random.uniform(0.85, 1.15)
    new_size = max(4, int(IMG_SIZE * scale))
    pil_scaled = Image.fromarray((arr * 255).astype(np.uint8)).resize(
        (new_size, new_size), Image.LANCZOS)
    small = np.array(pil_scaled, dtype=np.float32) / 255.0
    
    canvas = np.ones((IMG_SIZE, IMG_SIZE), dtype=np.float32)
    off = (IMG_SIZE - new_size) // 2
    sy, sx = max(0, off), max(0, off)
    ey = min(sy + small.shape[0], IMG_SIZE)
    ex = min(sx + small.shape[1], IMG_SIZE)
    canvas[sy:ey, sx:ex] = small[:ey - sy, :ex - sx]
    arr = canvas
    
    # Horizontal flip (50% chance)
    if random.random() > 0.5:
        arr = arr[:, ::-1].copy()
    
    # Gaussian noise
    arr += np.random.normal(0, 0.02, arr.shape).astype(np.float32)
    arr = np.clip(arr, 0.0, 1.0)
    
    return Image.fromarray((arr * 255).astype(np.uint8)).convert('L')

print('✅ Rendering functions ready')

## Step 5 — Generate glyphs for all characters

In [ ]:
def get_char_category(c):
    """Classify character into category."""
    if c in LOWERCASE:
        return 'lowercase'
    elif c in UPPERCASE:
        return 'uppercase'
    else:
        return 'punctuation'

stats = defaultdict(int)
glyph_id = 0

print(f'\n{"="*70}')
print('GENERATING SYNTHETIC WRITING ELEMENTS')
print(f'{"="*70}\n')

all_chars = LOWERCASE + UPPERCASE + PUNCTUATION

for char_idx, char in enumerate(all_chars):
    category = get_char_category(char)
    
    # Choose a random font for this character
    font_path = random.choice(FONT_PATHS) if FONT_PATHS else None
    
    # Render base character
    try:
        base_img = render_char(char, font_path=font_path)
    except Exception as e:
        print(f'  ⚠️  Error rendering {repr(char)}: {e}')
        stats['render_errors'] += 1
        continue
    
    # Generate augmented versions
    for aug_idx in range(AUGMENTATIONS_PER_CHAR):
        try:
            aug_img = augment_char(base_img, aug_idx)
            
            # Save
            out_path = os.path.join(OUTPUT_DIR, category, f'glyph_{glyph_id:08d}.png')
            aug_img.save(out_path)
            
            stats[category] += 1
            stats['total'] += 1
            glyph_id += 1
        except Exception as e:
            stats['aug_errors'] += 1
            continue
    
    if (char_idx + 1) % 10 == 0:
        pct = 100 * (char_idx + 1) // len(all_chars)
        print(f'  {pct:3d}% ({char_idx + 1}/{len(all_chars)})  glyphs generated: {stats["total"]:,}', flush=True)

print(f'\n{"="*70}')
print('GENERATION COMPLETE')
print(f'{"="*70}')
print(f'\nResults:')
print(f'  Lowercase: {stats["lowercase"]:,}')
print(f'  Uppercase: {stats["uppercase"]:,}')
print(f'  Punctuation: {stats["punctuation"]:,}')
print(f'  Total: {stats["total"]:,}')
if stats['render_errors']:
    print(f'  Render errors: {stats["render_errors"]}')
if stats['aug_errors']:
    print(f'  Augmentation errors: {stats["aug_errors"]}')

## Step 6 — Sample a few glyphs (sanity check)

In [ ]:
# Show a few random samples from each category
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax_idx, cat in enumerate(['lowercase', 'uppercase', 'punctuation']):
    cat_dir = os.path.join(OUTPUT_DIR, cat)
    imgs = [f for f in os.listdir(cat_dir) if f.endswith('.png')]
    
    if not imgs:
        axes[ax_idx].text(0.5, 0.5, f'{cat}\n(no glyphs)', ha='center', va='center')
        axes[ax_idx].set_title(cat)
        continue
    
    # Load a random sample
    sample_file = random.choice(imgs)
    sample_path = os.path.join(cat_dir, sample_file)
    img = Image.open(sample_path).convert('L')
    
    axes[ax_idx].imshow(img, cmap='gray')
    axes[ax_idx].set_title(f'{cat}\n({len(imgs)} glyphs)')
    axes[ax_idx].axis('off')

plt.tight_layout()
plt.show()

print('✅ Samples look good!')

## Step 7 — Organize and create zip

In [ ]:
# Move into glyphs/ structure to match training pipeline expectation
FINAL_DIR = '/content/writing_elements_structure'
GLYPHS_DIR = os.path.join(FINAL_DIR, 'glyphs')

if os.path.exists(FINAL_DIR):
    shutil.rmtree(FINAL_DIR)
os.makedirs(GLYPHS_DIR, exist_ok=True)

# Move categories
for cat in os.listdir(OUTPUT_DIR):
    src = os.path.join(OUTPUT_DIR, cat)
    dst = os.path.join(GLYPHS_DIR, cat)
    if os.path.isdir(src):
        shutil.move(src, dst)

print('Organized structure:')
for cat in sorted(os.listdir(GLYPHS_DIR)):
    cat_dir = os.path.join(GLYPHS_DIR, cat)
    n_imgs = len([f for f in os.listdir(cat_dir) if f.endswith('.png')])
    print(f'  {cat:15s}: {n_imgs:6,d} glyphs')

total = sum(
    len([f for f in os.listdir(os.path.join(GLYPHS_DIR, cat)) if f.endswith('.png')])
    for cat in os.listdir(GLYPHS_DIR)
)
print(f'\n✅ Total: {total:,} glyphs')

## Step 8 — Create zip file

In [ ]:
print(f'Creating zip: {ZIP_PATH}')

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(FINAL_DIR):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, FINAL_DIR)
            zf.write(file_path, arcname)

zip_size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
print(f'\n✅ Created {ZIP_PATH} ({zip_size_mb:.1f} MB)')

## Step 9 — Upload to Drive (optional)

In [ ]:
# Uncomment and edit path to upload to Drive
# import shutil
# DRIVE_PATH = '/content/gdrive/MyDrive/entrainement_ocr_echecs/writing_elements_classifier.zip'
# shutil.copy2(ZIP_PATH, DRIVE_PATH)
# print(f'✅ Uploaded to {DRIVE_PATH}')

print(f'To upload manually:')
print(f'  from google.colab import files')
print(f'  files.download("{ZIP_PATH}")')

## Step 10 — Download zip locally

In [ ]:
from google.colab import files
files.download(ZIP_PATH)
print(f'✅ Downloaded {ZIP_PATH}')